# GWAS Annotation Pipeline

**Purpose**: Annotate ~122,000 candidate genomic sites from a genome-wide selection scan with GWAS associations, assign derived-allele effect directions, classify traits using EFO ontology, and cluster redundant traits via Jaccard similarity.

**Pipeline steps**
0. Configuration & Imports
1. Load Input Data
2. Load GWAS Catalog
3. rsID Lookup (Ensembl REST API)
4. LD Proxy Lookup (Ensembl LD API / plink2 batch)
5. GWAS Catalog Annotation
6. Derived Allele Direction Assignment
7. EFO Trait Classification
8. Jaccard Trait Clustering
9. MHC Exclusion & Final Output


## 0. Configuration & Imports

All file paths and tunable parameters are defined here so the notebook can be adapted to a new dataset by editing only this cell.

In [ ]:
# ── Standard library ────────────────────────────────────────────────────────
import os
import re
import time
import json
import logging
from pathlib import Path
from itertools import combinations

# ── Third-party ──────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import requests
import networkx as nx

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
logger = logging.getLogger(__name__)

# ── Input / output paths ─────────────────────────────────────────────────────
SELECTION_SCAN_FILE  = 'Seletion_Stuff_ForHunter.txt'          # selection-scan input
GWAS_CATALOG_PKL     = '/mnt/datalake/gwas_catalog/gwas_catalog.pkl'
RSID_CACHE_FILE      = 'rsid_lookup_cache.json'                # cache for Ensembl rsID calls
LD_PROXY_FILE        = 'ld_proxy_table.csv'                    # plink2 LD output (assembled)
VCOR_FILE            = 'vcor_signed_r_table.csv'               # plink2 signed-r output
OUTPUT_MAIN          = '/mnt/results/ForBmni_Selection_IncDec_withDirection_noMHC_clusters.csv'
OUTPUT_PSYCH         = '/mnt/results/ForBmni_Psych_p1e5_withDirection_clusters.csv'

# ── Ensembl API ──────────────────────────────────────────────────────────────
ENSEMBL_OVERLAP_URL  = 'https://rest.ensembl.org/overlap/region/human/{chrom}:{pos}-{pos}'
ENSEMBL_LD_URL       = 'https://rest.ensembl.org/ld/human/{rsid}/1000GENOMES:phase_3:EUR'
ENSEMBL_HEADERS      = {'Content-Type': 'application/json'}
RSID_RATE_LIMIT_S    = 0.15   # seconds between rsID requests
LD_RATE_LIMIT_S      = 0.35   # seconds between LD requests

# ── LD parameters ────────────────────────────────────────────────────────────
LD_R2_THRESHOLD      = 0.8
LD_WINDOW_KB         = 500
LD_POPULATION        = '1000GENOMES:phase_3:EUR'

# ── GWAS filter ──────────────────────────────────────────────────────────────
GWAS_PVALUE_THRESHOLD = 5e-8

# ── MHC region (GRCh38) ──────────────────────────────────────────────────────
MHC_CHROM = 'chr6'
MHC_START = 25_000_000
MHC_END   = 35_000_000

# ── Jaccard clustering thresholds ────────────────────────────────────────────
JACCARD_BROAD = 0.1
JACCARD_FINE  = 0.3

print('Configuration loaded.')


## 1. Load Input Data

The selection-scan file is tab-separated with 123,746 rows. Each row represents a candidate site identified by a genome-wide selection scan. Key columns include `HG38_POS` (e.g. `chr1:946538`), allele frequency (`AF`), selection coefficient (`S`), selection p-value (`P_X`), FDR, and the PhyloP447 cross-species conservation score. The full dataset also carries ancestral (`ANC`), reference (`REF`), and alternate (`ALT`) allele columns used for derived-allele assignment.


In [ ]:
# Load the selection-scan file
scan_df = pd.read_csv(SELECTION_SCAN_FILE, sep='\t', low_memory=False)
logger.info('Selection scan loaded: %d rows, %d columns', *scan_df.shape)

# Standardise the HG38_POS column (ensure 'chr' prefix)
scan_df['HG38_POS'] = scan_df['HG38_POS'].astype(str)
if not scan_df['HG38_POS'].str.startswith('chr').all():
    scan_df['HG38_POS'] = 'chr' + scan_df['CHROM'].astype(str) + ':' + scan_df['POS'].astype(str)

# Deduplicate to unique positions for API queries
unique_positions = scan_df[['HG38_POS', 'CHROM', 'POS', 'REF', 'ALT', 'ANC']].drop_duplicates('HG38_POS')
logger.info('Unique positions: %d', len(unique_positions))

print(scan_df.head(3).to_string())


## 2. Load GWAS Catalog

The GWAS Catalog (Sollis et al. 2023) is loaded from a local pickle file containing 622,784 associations. It is immediately filtered to genome-wide significant associations (P ≤ 5×10⁻⁸) to reduce memory footprint and focus on well-replicated signals.


In [ ]:
# Load GWAS Catalog from local pickle
gwas_raw = pd.read_pickle(GWAS_CATALOG_PKL)
logger.info('GWAS Catalog loaded: %d rows', len(gwas_raw))

# Filter to genome-wide significant associations
gwas_raw['P-VALUE'] = pd.to_numeric(gwas_raw['P-VALUE'], errors='coerce')
gwas_gws = gwas_raw[gwas_raw['P-VALUE'] <= GWAS_PVALUE_THRESHOLD].copy()
logger.info('After GWS filter (P <= %.0e): %d rows', GWAS_PVALUE_THRESHOLD, len(gwas_gws))

# Build a fast lookup dict: rsID -> list of GWAS rows
gwas_by_rsid = gwas_gws.groupby('SNPS')
gwas_rsid_set = set(gwas_gws['SNPS'].dropna().unique())
logger.info('Unique rsIDs in GWS GWAS Catalog: %d', len(gwas_rsid_set))


## 3. rsID Lookup (Ensembl REST API)

Each unique hg38 position is queried against the Ensembl REST API overlap endpoint to retrieve the corresponding dbSNP rsID. Where multiple variants are reported at the same position, allele-matching (REF/ALT) is preferred; otherwise the first returned rsID is used. Results are cached to disk so the API is not re-queried on subsequent runs.


In [ ]:
def fetch_rsid(chrom: str, pos: int, ref: str, alt: str,
               session: requests.Session) -> str | None:
    """Query Ensembl overlap endpoint for a single position and return the best rsID."""
    url = ENSEMBL_OVERLAP_URL.format(chrom=chrom, pos=pos)
    params = {'feature': 'variation'}
    try:
        resp = session.get(url, headers=ENSEMBL_HEADERS, params=params, timeout=30)
        resp.raise_for_status()
        variants = resp.json()
    except Exception as exc:
        logger.warning('rsID lookup failed for %s:%s — %s', chrom, pos, exc)
        return None

    if not variants:
        return None

    # Prefer allele-matched variant
    ref_u, alt_u = str(ref).upper(), str(alt).upper()
    for v in variants:
        alleles = [a.upper() for a in v.get('alleles', [])]
        if ref_u in alleles and alt_u in alleles:
            return v.get('id')

    # Fall back to first rsID
    return variants[0].get('id')


# Load cache if it exists
if os.path.exists(RSID_CACHE_FILE):
    with open(RSID_CACHE_FILE) as fh:
        rsid_cache: dict[str, str | None] = json.load(fh)
    logger.info('Loaded rsID cache: %d entries', len(rsid_cache))
else:
    rsid_cache = {}

# Query positions not yet in cache
to_query = unique_positions[~unique_positions['HG38_POS'].isin(rsid_cache)]
logger.info('Positions to query: %d', len(to_query))

session = requests.Session()
for _, row in to_query.iterrows():
    pos_key = row['HG38_POS']
    rsid = fetch_rsid(
        chrom=str(row['CHROM']),
        pos=int(row['POS']),
        ref=row.get('REF', ''),
        alt=row.get('ALT', ''),
        session=session,
    )
    rsid_cache[pos_key] = rsid
    time.sleep(RSID_RATE_LIMIT_S)

# Persist cache
with open(RSID_CACHE_FILE, 'w') as fh:
    json.dump(rsid_cache, fh)
logger.info('rsID cache saved: %d entries', len(rsid_cache))

# Attach rsIDs to the unique-positions table
unique_positions = unique_positions.copy()
unique_positions['Input_rsID'] = unique_positions['HG38_POS'].map(rsid_cache)
mapped = unique_positions['Input_rsID'].notna().sum()
logger.info('Positions with rsID: %d / %d', mapped, len(unique_positions))


## 4. LD Proxy Lookup (Ensembl LD API / plink2 batch)

### 4a. Ensembl LD API (small-scale / validation)
For each input rsID, the Ensembl LD REST API is queried to retrieve all variants with R² ≥ 0.8 in the EUR 1000 Genomes Phase 3 population within a ±500 kb window.

### 4b. plink2 batch approach (full 122K dataset)
For the full dataset, per-chromosome high-coverage phased VCF files from the 1000 Genomes Project (GRCh38, 3,202 samples; Byrska-Bishop et al. 2022) were used. plink2 was run with `--r2-unphased` (R²) and `--r-unphased` (signed r) to compute LD in batch. The resulting files are loaded here and assembled into a proxy lookup table.


In [ ]:
# ── 4a. Ensembl LD API helper (used for small batches / validation) ──────────

def fetch_ld_proxies_api(rsid: str, session: requests.Session,
                          r2_threshold: float = LD_R2_THRESHOLD,
                          window_kb: int = LD_WINDOW_KB) -> list[dict]:
    """Return LD proxies for *rsid* from the Ensembl LD REST API.

    Always includes the query variant itself (R²=1.0, r=1.0).
    Retries once on HTTP 429 (rate-limit) after a 60-second pause.
    """
    url = ENSEMBL_LD_URL.format(rsid=rsid)
    params = {'r2': r2_threshold, 'window_size': window_kb}
    for attempt in range(2):
        try:
            resp = session.get(url, headers=ENSEMBL_HEADERS, params=params, timeout=60)
            if resp.status_code == 429:
                logger.warning('Rate-limited on %s; sleeping 60 s', rsid)
                time.sleep(60)
                continue
            resp.raise_for_status()
            proxies = resp.json()
            break
        except Exception as exc:
            logger.warning('LD lookup failed for %s (attempt %d): %s', rsid, attempt + 1, exc)
            proxies = []
            break

    results = []
    for p in proxies:
        results.append({
            'Input_rsID':  rsid,
            'Proxy_rsID':  p.get('variation2') if p.get('variation1') == rsid else p.get('variation1'),
            'LD_R2':       float(p.get('r2', 0)),
            'LD_r':        float(p.get('r', 1)),   # signed r (may not be returned by all endpoints)
        })

    # Always include the query variant itself as a direct hit
    results.append({'Input_rsID': rsid, 'Proxy_rsID': rsid, 'LD_R2': 1.0, 'LD_r': 1.0})
    return results


# ── 4b. Load plink2 batch LD output ──────────────────────────────────────────

# Expected columns from assembled plink2 output:
#   Input_rsID, Proxy_rsID, LD_R2, LD_r  (signed r from --r-unphased)
ld_proxy_df = pd.read_csv(LD_PROXY_FILE, low_memory=False)
logger.info('LD proxy table loaded: %d rows', len(ld_proxy_df))

# Ensure required columns are present
assert {'Input_rsID', 'Proxy_rsID', 'LD_R2', 'LD_r'}.issubset(ld_proxy_df.columns), \
    'LD proxy table is missing required columns'

# Filter to R² >= threshold (should already be filtered, but enforce here)
ld_proxy_df = ld_proxy_df[ld_proxy_df['LD_R2'] >= LD_R2_THRESHOLD].copy()

# Build a dict: Input_rsID -> DataFrame of proxies
ld_proxy_by_input = {rsid: grp for rsid, grp in ld_proxy_df.groupby('Input_rsID')}
logger.info('Input rsIDs with LD proxies: %d', len(ld_proxy_by_input))


## 5. GWAS Catalog Annotation

For each candidate site, all proxy rsIDs (R² ≥ 0.8) are looked up in the genome-wide significant GWAS Catalog. Matched records are joined back to the input position table, producing a long-format table of (input_position × GWAS_trait) pairs.


In [ ]:
# Merge unique_positions with LD proxy table on Input_rsID
pos_with_rsid = unique_positions.dropna(subset=['Input_rsID'])
pos_ld = pos_with_rsid.merge(ld_proxy_df, on='Input_rsID', how='left')

# For positions with no LD proxies, treat the input rsID itself as the only proxy
no_proxy_mask = pos_ld['Proxy_rsID'].isna()
pos_ld.loc[no_proxy_mask, 'Proxy_rsID'] = pos_ld.loc[no_proxy_mask, 'Input_rsID']
pos_ld.loc[no_proxy_mask, 'LD_R2']     = 1.0
pos_ld.loc[no_proxy_mask, 'LD_r']      = 1.0

# Rename GWAS Catalog columns for clarity
gwas_rename = {
    'SNPS':                    'Proxy_rsID',
    'DISEASE/TRAIT':           'Trait',
    'P-VALUE':                 'P_value',
    'OR or BETA':              'OR_or_Beta',
    '95% CI (TEXT)':           'CI_95',
    'RISK ALLELE FREQUENCY':   'Risk_Allele_Freq',
    'STRONGEST SNP-RISK ALLELE': 'Risk_Allele_Raw',
    'MAPPED_GENE':             'Mapped_Gene',
    'FIRST AUTHOR':            'First_Author',
    'JOURNAL':                 'Journal',
    'PUBMEDID':                'PubMed_ID',
    'CHR_ID':                  'GWAS_CHR',
    'CHR_POS':                 'GWAS_POS',
}
gwas_slim = gwas_gws.rename(columns=gwas_rename)[list(gwas_rename.values())].copy()

# Join: input positions × proxies × GWAS associations
annotated = pos_ld.merge(gwas_slim, on='Proxy_rsID', how='inner')
logger.info('Annotated associations: %d rows', len(annotated))

# Add convenience columns
annotated['Is_Direct_Hit'] = annotated['LD_R2'] == 1.0
annotated['GWAS_rsID']     = annotated['Proxy_rsID']

# Construct Proxy_Position_HG38 from GWAS CHR/POS
annotated['Proxy_Position_HG38'] = (
    'chr' + annotated['GWAS_CHR'].astype(str) + ':' + annotated['GWAS_POS'].astype(str)
)

print(f'Annotated table shape: {annotated.shape}')
print(annotated[['HG38_POS', 'Input_rsID', 'Proxy_rsID', 'LD_R2', 'Trait', 'P_value']].head(5).to_string())


## 6. Derived Allele Direction Assignment

For each (input_position × GWAS_trait) pair, the direction of the **derived allele** on the trait is inferred in three sub-steps:

1. **Parse risk allele** from the `STRONGEST SNP-RISK ALLELE` field (format `rsXXXXXX-A`).
2. **Determine risk direction** from OR/Beta: >1 → increases; <1 → decreases; ambiguous → `unknown_risk_direction`.
3. **Assign derived-allele direction**: compare the derived allele (ALT if ANC=REF, else REF) to the risk allele, accounting for the sign of the LD correlation (r) for proxy hits.


In [ ]:
# ── 6a. Parse risk allele ────────────────────────────────────────────────────

def parse_risk_allele(raw: str) -> str | None:
    """Extract the allele base from a GWAS Catalog 'rsXXXXXX-A' string."""
    if pd.isna(raw):
        return None
    m = re.search(r'-([ACGT?]+)$', str(raw).strip())
    return m.group(1).upper() if m else None


annotated['Risk_Allele'] = annotated['Risk_Allele_Raw'].apply(parse_risk_allele)


# ── 6b. Determine risk direction from OR / Beta ───────────────────────────────

def risk_direction(or_beta, ci_text: str) -> str:
    """Return 'increases', 'decreases', or 'unknown_risk_direction'."""
    ci_text = str(ci_text).lower() if not pd.isna(ci_text) else ''
    try:
        val = float(or_beta)
    except (TypeError, ValueError):
        # Fall back to CI text
        if 'increase' in ci_text:
            return 'increases'
        if 'decrease' in ci_text:
            return 'decreases'
        return 'unknown_risk_direction'

    if val > 1:
        return 'increases'
    if val < 1:
        return 'decreases'
    # OR == 1 exactly — check CI text
    if 'increase' in ci_text:
        return 'increases'
    if 'decrease' in ci_text:
        return 'decreases'
    return 'unknown_risk_direction'


annotated['Risk_Direction'] = annotated.apply(
    lambda r: risk_direction(r['OR_or_Beta'], r['CI_95']), axis=1
)


# ── 6c. Identify derived allele ───────────────────────────────────────────────

def derived_allele(ref: str, alt: str, anc: str) -> str | None:
    """Return the derived allele: ALT if ANC==REF, REF if ANC==ALT, else None."""
    ref, alt, anc = str(ref).upper(), str(alt).upper(), str(anc).upper()
    if anc == ref:
        return alt
    if anc == alt:
        return ref
    return None  # ancestral allele ambiguous


annotated['Derived_Allele'] = annotated.apply(
    lambda r: derived_allele(r['REF'], r['ALT'], r['ANC']), axis=1
)


# ── 6d. Assign derived-allele direction ───────────────────────────────────────

def derived_allele_direction(row) -> str:
    """Assign the direction of the derived allele's effect on the trait.

    For proxy hits (R² < 1), the sign of the LD correlation (r) is used to
    account for potential allele-coding flips between the input and proxy variants.
    """
    risk_dir   = row['Risk_Direction']
    derived    = row['Derived_Allele']
    risk_allele = row['Risk_Allele']
    ld_r       = row.get('LD_r', 1.0)

    if risk_dir == 'unknown_risk_direction':
        return 'unknown_risk_direction'
    if pd.isna(derived) or pd.isna(risk_allele):
        return 'unknown'

    derived_matches_risk = (derived.upper() == risk_allele.upper())

    # For proxy hits with negative r, the allele coding is flipped
    if not row['Is_Direct_Hit'] and float(ld_r) < 0:
        derived_matches_risk = not derived_matches_risk

    if derived_matches_risk:
        return risk_dir          # 'increases' or 'decreases'
    else:
        return 'decreases' if risk_dir == 'increases' else 'increases'


annotated['Derived_Allele_Direction'] = annotated.apply(derived_allele_direction, axis=1)

direction_counts = annotated['Derived_Allele_Direction'].value_counts()
print('Derived allele direction counts:')
print(direction_counts.to_string())


## 7. EFO Trait Classification

Each unique GWAS trait string is classified into two levels of ontological granularity using a keyword-based classifier informed by the Experimental Factor Ontology (EFO; Malone et al. 2010). `EFO_fine` assigns one of 14 psychiatric/neurological/behavioural categories; `EFO_broad` groups traits into higher-order categories (e.g. Psychiatric & Cognitive, Metabolic, Cardiovascular). Classification is performed by applying regular expressions to the trait name string.


In [ ]:
# ── EFO fine-level classification rules ──────────────────────────────────────
# Each entry: (EFO_fine label, compiled regex pattern)
EFO_FINE_RULES: list[tuple[str, re.Pattern]] = [
    ('Cognitive Ability & Intelligence',
     re.compile(r'cogniti|intelligen|IQ|educational attain|fluid intel|g factor|working memory|'
                r'processing speed|verbal|reasoning|executive function', re.I)),
    ('Schizophrenia & Psychosis',
     re.compile(r'schizophreni|psychos|psychotic', re.I)),
    ('Bipolar Disorder',
     re.compile(r'bipolar|manic|mania', re.I)),
    ('Depression & Mood',
     re.compile(r'depress|major depressive|MDD|mood disorder|dysthymi', re.I)),
    ('Anxiety & Neuroticism',
     re.compile(r'anxiety|neurotic|worry|panic disorder|phobia', re.I)),
    ('ADHD',
     re.compile(r'\bADHD\b|attention.deficit|hyperactiv', re.I)),
    ('Autism Spectrum',
     re.compile(r'autism|autistic|ASD\b|asperger', re.I)),
    ('Sleep',
     re.compile(r'sleep|insomnia|chronotype|circadian|narcolep|daytime sleepiness', re.I)),
    ('Personality & Well-being',
     re.compile(r'personality|well.being|life satisfaction|subjective well|openness|'
                r'conscientiousness|extraversion|agreeableness|neuroticism', re.I)),
    ('Brain Imaging',
     re.compile(r'brain (volume|structure|imaging|MRI|connectivity|white matter|grey matter|'
                r'cortical|subcortical)|hippocampal volume|amygdala', re.I)),
    ('Other Neurological',
     re.compile(r'alzheimer|parkinson|epilep|multiple sclerosis|migraine|stroke|'
                r'neurological|dementia|ALS\b|amyotrophic', re.I)),
    ('Alcohol Consumption',
     re.compile(r'alcohol|drinks per week|alcohol use disorder|AUD\b', re.I)),
    ('Smoking Behavior',
     re.compile(r'smok|cigarette|nicotine|tobacco', re.I)),
    ('Other Psychiatric',
     re.compile(r'psychiatric|mental|eating disorder|anorexia|bulimia|OCD\b|'
                r'obsessive.compulsive|PTSD|post.traumatic|conduct disorder', re.I)),
]

# ── EFO broad-level classification rules ─────────────────────────────────────
EFO_BROAD_RULES: list[tuple[str, re.Pattern]] = [
    ('Psychiatric & Cognitive',
     re.compile(r'cogniti|intelligen|IQ|educational|schizophreni|psychos|bipolar|manic|'
                r'depress|anxiety|neurotic|ADHD|attention.deficit|autism|autistic|ASD\b|'
                r'sleep|insomnia|chronotype|personality|well.being|brain|hippocampal|'
                r'alzheimer|parkinson|epilep|migraine|dementia|alcohol|smok|cigarette|'
                r'psychiatric|mental|eating disorder|anorexia|OCD\b|PTSD', re.I)),
    ('Metabolic',
     re.compile(r'BMI|body mass|obesity|diabet|insulin|glucose|lipid|cholesterol|'
                r'triglycerid|metabol|adipos|waist|fat mass', re.I)),
    ('Cardiovascular',
     re.compile(r'coronary|cardiac|heart|blood pressure|hypertension|atrial|'
                r'myocardial|stroke|cardiovascular|QRS|QT interval', re.I)),
    ('Immune & Inflammatory',
     re.compile(r'immune|autoimmune|inflamm|rheumatoid|lupus|crohn|colitis|'
                r'asthma|allerg|eczema|psoriasis|multiple sclerosis', re.I)),
    ('Anthropometric',
     re.compile(r'height|weight|body composition|lean mass|bone density|'
                r'hand grip|birth weight', re.I)),
    ('Reproductive & Hormonal',
     re.compile(r'testosterone|estrogen|oestrogen|menopause|menarche|fertility|'
                r'reproductive|sex hormone|puberty|birth', re.I)),
    ('Cancer',
     re.compile(r'cancer|carcinoma|tumor|tumour|melanoma|leukemia|lymphoma|'
                r'glioma|breast cancer|prostate cancer|lung cancer', re.I)),
    ('Other',
     re.compile(r'.*', re.I)),  # catch-all
]


def classify_trait(trait: str, rules: list[tuple[str, re.Pattern]]) -> str:
    """Return the first matching category label for *trait*, or 'Unclassified'."""
    for label, pattern in rules:
        if pattern.search(str(trait)):
            return label
    return 'Unclassified'


# Classify each unique trait string (then map back to avoid redundant computation)
unique_traits = annotated['Trait'].dropna().unique()
trait_efo_fine  = {t: classify_trait(t, EFO_FINE_RULES)  for t in unique_traits}
trait_efo_broad = {t: classify_trait(t, EFO_BROAD_RULES) for t in unique_traits}

annotated['EFO_fine']  = annotated['Trait'].map(trait_efo_fine)
annotated['EFO_broad'] = annotated['Trait'].map(trait_efo_broad)

print('EFO_fine distribution:')
print(annotated['EFO_fine'].value_counts().to_string())
print('\nEFO_broad distribution:')
print(annotated['EFO_broad'].value_counts().to_string())


In [ ]:
def jaccard_similarity(set_a: set, set_b: set) -> float:
    """Compute Jaccard similarity between two sets."""
    if not set_a and not set_b:
        return 0.0
    return len(set_a & set_b) / len(set_a | set_b)


def build_trait_clusters(df: pd.DataFrame,
                          jaccard_threshold: float,
                          pos_col: str = 'HG38_POS',
                          trait_col: str = 'Trait') -> pd.DataFrame:
    """Cluster traits by Jaccard similarity of their associated positions.

    Returns a DataFrame with columns:
        Trait, Cluster_ID, Cluster_Size, Is_Representative, Representative_Trait
    """
    # Build trait -> set of positions
    trait_pos: dict[str, set] = (
        df.groupby(trait_col)[pos_col]
          .apply(set)
          .to_dict()
    )
    traits = list(trait_pos.keys())
    n_assoc = {t: len(s) for t, s in trait_pos.items()}

    # Build similarity graph
    G = nx.Graph()
    G.add_nodes_from(traits)
    for t1, t2 in combinations(traits, 2):
        j = jaccard_similarity(trait_pos[t1], trait_pos[t2])
        if j >= jaccard_threshold:
            G.add_edge(t1, t2, weight=j)

    # Connected components → cluster IDs
    cluster_records = []
    for cluster_id, component in enumerate(nx.connected_components(G)):
        component = list(component)
        cluster_size = len(component)
        # Representative = trait with most associations
        representative = max(component, key=lambda t: n_assoc.get(t, 0))
        for trait in component:
            cluster_records.append({
                trait_col:           trait,
                'Cluster_ID':        cluster_id,
                'Cluster_Size':      cluster_size,
                'Is_Representative': trait == representative,
                'Representative_Trait': representative,
            })

    return pd.DataFrame(cluster_records)


# ── Broad clusters (Jaccard >= 0.1) ──────────────────────────────────────────
logger.info('Computing broad Jaccard clusters (threshold=%.1f)...', JACCARD_BROAD)
broad_clusters = build_trait_clusters(annotated, jaccard_threshold=JACCARD_BROAD)
broad_clusters = broad_clusters.rename(columns={
    'Cluster_ID':          'Cluster_ID',
    'Cluster_Size':        'Cluster_Size',
    'Is_Representative':   'Is_Representative',
    'Representative_Trait':'Representative_Trait',
})
annotated = annotated.merge(broad_clusters, on='Trait', how='left')
logger.info('Broad clusters: %d unique cluster IDs', annotated['Cluster_ID'].nunique())

print('Cluster size distribution (broad):')
print(annotated['Cluster_Size'].value_counts().head(10).to_string())


In [ ]:
# ── Flag MHC variants ────────────────────────────────────────────────────────
def is_mhc(hg38_pos: str) -> bool:
    """Return True if the position falls within the MHC region (chr6:25Mb-35Mb)."""
    try:
        chrom, pos_str = hg38_pos.split(':')
        return chrom == MHC_CHROM and MHC_START <= int(pos_str) <= MHC_END
    except Exception:
        return False


annotated['In_MHC'] = annotated['HG38_POS'].apply(is_mhc)
logger.info('MHC variants: %d', annotated['In_MHC'].sum())

# ── Re-cluster on MHC-excluded data ──────────────────────────────────────────
annotated_noMHC = annotated[~annotated['In_MHC']].copy()
logger.info('Computing noMHC Jaccard clusters (threshold=%.1f)...', JACCARD_BROAD)
noMHC_clusters = build_trait_clusters(annotated_noMHC, jaccard_threshold=JACCARD_BROAD)
noMHC_clusters = noMHC_clusters.rename(columns={
    'Cluster_ID':          'Cluster_ID_noMHC',
    'Cluster_Size':        'Cluster_Size_noMHC',
    'Is_Representative':   'Is_Representative_noMHC',
    'Representative_Trait':'Representative_Trait_noMHC',
})

# Merge noMHC cluster columns back onto the full annotated table
annotated = annotated.merge(noMHC_clusters, on='Trait', how='left')
logger.info('noMHC clusters: %d unique cluster IDs', annotated['Cluster_ID_noMHC'].nunique())

# ── Merge back selection-scan metadata ───────────────────────────────────────
scan_meta = scan_df[['HG38_POS', 'AF', 'FDR', 'PhyloP447',
                      'NearestGeneWrong', 'NearestDistWrong']].drop_duplicates('HG38_POS')
scan_meta = scan_meta.rename(columns={
    'AF':              'Input_AF',
    'FDR':             'Input_FDR',
    'PhyloP447':       'Input_PhyloP447',
    'NearestGeneWrong':'Nearest_Gene',
})
annotated = annotated.merge(scan_meta[['HG38_POS', 'Input_AF', 'Input_FDR',
                                        'Input_PhyloP447', 'Nearest_Gene']],
                             on='HG38_POS', how='left')

# ── Select and order final columns ───────────────────────────────────────────
FINAL_COLS = [
    'HG38_POS', 'ANC', 'REF', 'ALT', 'Derived_Allele', 'Nearest_Gene',
    'Input_AF', 'Input_FDR', 'Input_PhyloP447',
    'Proxy_Position_HG38', 'LD_R2', 'Is_Direct_Hit', 'LD_Coverage',
    'Trait', 'P_value', 'Risk_Allele', 'OR_or_Beta', 'CI_95',
    'GWAS_rsID', 'Mapped_Gene', 'First_Author', 'Journal', 'PubMed_ID',
    'Cluster_ID', 'Cluster_Size', 'Is_Representative', 'Representative_Trait',
    'Risk_Direction', 'Derived_Allele_Direction',
    'Cluster_ID_noMHC', 'Cluster_Size_noMHC',
    'Is_Representative_noMHC', 'Representative_Trait_noMHC',
    'EFO_fine', 'EFO_broad',
]

# Add LD_Coverage (fraction of input positions with at least one LD proxy)
n_with_proxy = annotated['HG38_POS'].isin(
    annotated.loc[annotated['LD_R2'] < 1.0, 'HG38_POS']
).sum()
annotated['LD_Coverage'] = n_with_proxy / len(annotated)

# Keep only columns that exist
final_cols_present = [c for c in FINAL_COLS if c in annotated.columns]
output_df = annotated[final_cols_present].copy()

# ── Write main output ─────────────────────────────────────────────────────────
output_df.to_csv(OUTPUT_MAIN, index=False)
logger.info('Main output written: %s  (%d rows × %d cols)',
            OUTPUT_MAIN, *output_df.shape)

# ── Write Psychiatric & Cognitive subset ─────────────────────────────────────
psych_df = output_df[
    (output_df['EFO_broad'] == 'Psychiatric & Cognitive') &
    (output_df['P_value'] <= 1e-5)
].copy()
psych_df.to_csv(OUTPUT_PSYCH, index=False)
logger.info('Psychiatric subset written: %s  (%d rows × %d cols)',
            OUTPUT_PSYCH, *psych_df.shape)

print(f'\nFinal output: {output_df.shape[0]:,} rows × {output_df.shape[1]} columns')
print(f'Psychiatric subset: {psych_df.shape[0]:,} rows × {psych_df.shape[1]} columns')
print('\nColumn list:')
print(output_df.columns.tolist())
